In [164]:
import pandas as pd
import ast
from sklearn.model_selection import GroupKFold
import numpy as np
import os
import re
from typing import Dict, Tuple, Optional
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import MultiLabelBinarizer


# categorical

In [21]:
annot_texts = pd.read_csv("./data/annotated/full_ner/all_annotations.csv")[['doc_id','Text']]
annot_texts

,doc_id,Text
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...
...,...,...
1609,My_pdf411new_method,\r\n2. Material and methods\r\n2.1. Animals\r\...
1610,My_pdf726_title^abstract,Unique Properties Associated with the Brain Pe...
1611,My_pdf726_method,\r\nMethods\r\nMice\r\nAll animal procedures w...
1612,My_pdf325new_title^abstract,Effects of microRNA-10a on synapse remodeling ...


In [46]:
def run_classifier_on_chunk(
    df_chunk: pd.DataFrame,
    text_col: str,
    category_name: str,
    clf_obj,
    format_fn: Optional[callable],
) -> pd.DataFrame:
    """Run a single classifier on df_chunk[text_col], returning a DataFrame with new columns."""
    # Work on a view to avoid copying large frames unnecessarily
    def apply_and_format(txt):
        raw_out = clf_obj.classify(txt)
        return format_fn(raw_out) if format_fn else raw_out

    if category_name == "assay":
        df_chunk[["prediction_encoded_num", "prediction_encoded_label", "prediction_tokens"]] = (
            df_chunk[text_col].apply(lambda txt: pd.Series(apply_and_format(txt)))
        )
    else:
        df_chunk[["prediction_encoded_num", "prediction_encoded_label"]] = (
            df_chunk[text_col].apply(lambda txt: pd.Series(apply_and_format(txt)))
        )
    return df_chunk

In [141]:
def evaluate_label_predictions(df, true_col="label", pred_col="prediction_encoded_label"):
    """
    Evaluate predicted vs. true labels using Precision, Recall, and F1 scores.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain two columns:
        - true_col: the ground-truth labels (strings)
        - pred_col: the predicted labels (strings)
    true_col : str
        Name of the column with true labels.
    pred_col : str
        Name of the column with predicted labels.

    Returns
    -------
    dict
        A dictionary containing precision, recall, F1, and the classification report.
    """

    # --- Prepare labels ---
    y_true = df[true_col].astype(str).str.strip()
    y_pred = df[pred_col].astype(str).str.strip()

    # --- Ensure label consistency ---
    all_labels = sorted(set(y_true) | set(y_pred))

    # --- Compute metrics ---
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    report = classification_report(y_true, y_pred, target_names=all_labels, zero_division=0)

    # --- Print summary ---
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F1 Score:  {f1:.3f}\n")
    print("Classification Report:\n", report)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "report": report
    }

## sex

In [127]:
def preprocess_label_column(df, label_col="label"):
    """
    Clean and standardize the label column:
    - Ensure each cell is treated as a list.
    - Convert list-like strings to real lists.
    - If multiple values are present, prefer specific ones (e.g., 'sex-female' or 'sex-male')
      over 'sex-not-reported' or 'unknown'.
    - Output is a single clean string label.
    """
    df = df.copy()

    def to_list_safe(x):
        """Convert strings or NaNs to proper lists safely."""
        if pd.isna(x):
            return []
        if isinstance(x, (list, tuple)):
            return list(x)
        if isinstance(x, str):
            x = x.strip()
            # If it looks like a list in string form, try to parse it
            if x.startswith("[") and x.endswith("]"):
                try:
                    parsed = ast.literal_eval(x)
                    if isinstance(parsed, list):
                        return parsed
                except Exception:
                    pass
            # Otherwise just treat it as a single label
            return [x]
        return [str(x)]

    def choose_label(labels):
        """Select the most specific label if multiple exist."""
        labels = [str(l).strip() for l in labels if l and str(l).strip().lower() != "nan"]
        if not labels:
            return "unknown"

        # Priority order
        priority = ["sex-male", "sex-female", "sex-unknown", "sex-not-reported"]

        for p in priority:
            if p in labels:
                return p

        # Otherwise just return the first label
        return labels[0]

    df[label_col] = df[label_col].apply(lambda x: choose_label(to_list_safe(x)))
    return df

In [129]:
sys.path.append(os.path.abspath("../08_IE_full_text/regex_classifiers"))
from sex_classifier import SexClassifier

In [131]:
sex_df = pd.read_csv("./data/annotated/full_categorical/sex_annotations.csv")
sex_df = sex_df.rename(columns={"label_id":"doc_id"})
sex_df = sex_df.merge(annot_texts, on="doc_id", how="left")

In [133]:
sex_df = preprocess_label_column(sex_df)
sex_df.head()

,doc_id,annotations_array_numeric,annotator,label,Text
0,My_pdf110new_method,[0 1 0 0],annot1,sex-male,\r\nMATERIALS AND METHODS\r\nExperimental anim...
1,My_pdf110new_title^abstract,[0 1 0 0],annot1,sex-male,Effects of different frequencies of electro-ac...
2,My_pdf118new_method,[0 1 0 0],annot1,sex-male,\r\nMaterials and Methods\r\nAneurysm Construc...
3,My_pdf118new_title^abstract,[0 0 0 1],annot1,sex-not-reported,Endovascular treatment of intracranial aneurys...
4,My_pdf127new_method,[0 0 1 0],annot1,sex-both,\r\nMaterials and Methods\r\niPS cell cultures...


In [135]:
sex_classifier = SexClassifier()

In [137]:
sex_annotated = run_classifier_on_chunk(sex_df, "Text", "sex", sex_classifier, format_fn=None)

In [138]:
sex_annotated

,doc_id,annotations_array_numeric,annotator,label,Text,prediction_encoded_num,prediction_encoded_label
0,My_pdf110new_method,[0 1 0 0],annot1,sex-male,\r\nMATERIALS AND METHODS\r\nExperimental anim...,2,sex-male
1,My_pdf110new_title^abstract,[0 1 0 0],annot1,sex-male,Effects of different frequencies of electro-ac...,2,sex-male
2,My_pdf118new_method,[0 1 0 0],annot1,sex-male,\r\nMaterials and Methods\r\nAneurysm Construc...,2,sex-male
3,My_pdf118new_title^abstract,[0 0 0 1],annot1,sex-not-reported,Endovascular treatment of intracranial aneurys...,0,sex-not-reported
4,My_pdf127new_method,[0 0 1 0],annot1,sex-both,\r\nMaterials and Methods\r\niPS cell cultures...,3,sex-both
...,...,...,...,...,...,...,...
1445,My_pdf860_title^abstract,[0 0 0 1],overlapping,sex-not-reported,Combined Nurr1 and Foxa2 roles in the therapy ...,0,sex-not-reported
1446,My_pdf876_method,[0 1 0 0],overlapping,sex-male,\r\n2 Material and methods\r\n2.1 Animal and r...,2,sex-male
1447,My_pdf876_title^abstract,[0 1 0 0],overlapping,sex-male,Efficacy of Danhong Injection on Serum Concent...,2,sex-male
1448,My_pdf881_method,[0 1 0 0],overlapping,sex-male,\r\nMaterials and Methods\r\nReagents\r\nThe a...,2,sex-male


In [143]:
evaluate_label_predictions(sex_annotated)

Precision: 0.956
Recall:    0.950
F1 Score:  0.952

Classification Report:
                   precision    recall  f1-score   support

        sex-both       0.53      0.71      0.61        63
      sex-female       0.94      0.86      0.90       135
        sex-male       0.99      0.96      0.97       525
sex-not-reported       0.97      0.98      0.98       727

        accuracy                           0.95      1450
       macro avg       0.86      0.88      0.86      1450
    weighted avg       0.96      0.95      0.95      1450



{'precision': 0.9562471709361505,
 'recall': 0.9496551724137932,
 'f1': 0.9521245489002789,
 'report': '                  precision    recall  f1-score   support\n\n        sex-both       0.53      0.71      0.61        63\n      sex-female       0.94      0.86      0.90       135\n        sex-male       0.99      0.96      0.97       525\nsex-not-reported       0.97      0.98      0.98       727\n\n        accuracy                           0.95      1450\n       macro avg       0.86      0.88      0.86      1450\n    weighted avg       0.96      0.95      0.95      1450\n'}

## species

In [170]:
def evaluate_multilabel_predictions(df, true_col="label", pred_col="prediction_encoded_label"):
    """
    Evaluate multi-label classification performance (Precision, Recall, F1, and per-label report)
    using string or array-based labels.

    Works even if cells contain numpy arrays, Python lists, or stringified lists.
    """

    df = df.copy()

    def to_list_safe(x):
        """Normalize each cell to a Python list of string labels."""
        # Handle NumPy arrays first (avoid ValueError)
        if isinstance(x, np.ndarray):
            x = x.tolist()

        # Handle missing or empty values
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return []

        # Handle already-list values
        if isinstance(x, (list, tuple)):
            return [str(i).strip() for i in x if str(i).strip()]

        # Handle stringified lists (e.g. "['rat', 'dog']")
        if isinstance(x, str):
            s = x.strip()
            if s.startswith("[") and s.endswith("]"):
                s = s.strip("[]").replace("'", "").replace('"', "")
                return [i.strip() for i in s.split(",") if i.strip()]
            elif s:
                return [s]

        # Default: wrap scalar as list
        return [str(x)]

    # Normalize both columns
    df["true_labels"] = df[true_col].apply(to_list_safe)
    df["pred_labels"] = df[pred_col].apply(to_list_safe)

    # Collect all unique labels
    all_labels = sorted(set([l for sub in df["true_labels"] for l in sub]) |
                        set([l for sub in df["pred_labels"] for l in sub]))

    # Use MultiLabelBinarizer for one-hot conversion
    mlb = MultiLabelBinarizer(classes=all_labels)
    y_true = mlb.fit_transform(df["true_labels"])
    y_pred = mlb.transform(df["pred_labels"])

    # Compute metrics
    precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    recall_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)

    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    # Generate per-class report
    report = classification_report(
        y_true, y_pred, target_names=all_labels, zero_division=0
    )

    # Print summary
    print(f"Micro  - Precision: {precision_micro:.3f}, Recall: {recall_micro:.3f}, F1: {f1_micro:.3f}")
    print(f"Macro  - Precision: {precision_macro:.3f}, Recall: {recall_macro:.3f}, F1: {f1_macro:.3f}\n")
    print("Classification Report:\n")
    print(report)

    return {
        "labels": all_labels,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "report": report,
    }

In [154]:
sys.path.append(os.path.abspath("../08_IE_full_text/regex_classifiers"))
from species_classifier import SpeciesClassifier

In [156]:
species_classifier = SpeciesClassifier()

In [150]:
species_df = pd.read_csv("./data/annotated/full_categorical/species_annotations.csv")
species_df = species_df.rename(columns={"label_id":"doc_id"})
species_df = species_df.merge(annot_texts, on="doc_id", how="left")

In [152]:
species_df

,doc_id,annotations_array_numeric,annotator,label,Text
0,My_pdf110new_method,[1 0 0 0 0 0 0 0 0],annot1,['rat'],\r\nMATERIALS AND METHODS\r\nExperimental anim...
1,My_pdf110new_title^abstract,[1 0 0 0 0 0 0 0 0],annot1,['rat'],Effects of different frequencies of electro-ac...
2,My_pdf118new_method,[0 0 0 0 1 0 0 0 0],annot1,['dog'],\r\nMaterials and Methods\r\nAneurysm Construc...
3,My_pdf118new_title^abstract,[0 0 0 0 1 0 0 0 0],annot1,['dog'],Endovascular treatment of intracranial aneurys...
4,My_pdf127new_method,[0 1 0 0 0 0 0 0 0],annot1,['mouse'],\r\nMaterials and Methods\r\niPS cell cultures...
...,...,...,...,...,...
1423,My_pdf860_title^abstract,[0 1 0 0 0 0 0 0 0],overlapping,['mouse'],Combined Nurr1 and Foxa2 roles in the therapy ...
1424,My_pdf876_method,[1 0 0 0 0 0 0 0 0],overlapping,['rat'],\r\n2 Material and methods\r\n2.1 Animal and r...
1425,My_pdf876_title^abstract,[1 0 0 0 0 0 0 0 0],overlapping,['rat'],Efficacy of Danhong Injection on Serum Concent...
1426,My_pdf881_method,[0 1 0 0 0 0 0 0 0],overlapping,['mouse'],\r\nMaterials and Methods\r\nReagents\r\nThe a...


In [158]:
species_annotated = run_classifier_on_chunk(species_df, "Text", "sex", species_classifier, format_fn=None)
species_annotated

,doc_id,annotations_array_numeric,annotator,label,Text,prediction_encoded_num,prediction_encoded_label
0,My_pdf110new_method,[1 0 0 0 0 0 0 0 0],annot1,['rat'],\r\nMATERIALS AND METHODS\r\nExperimental anim...,"[0, 1, 0, 0, 0, 0, 0, 0, 0]",[rat]
1,My_pdf110new_title^abstract,[1 0 0 0 0 0 0 0 0],annot1,['rat'],Effects of different frequencies of electro-ac...,"[0, 1, 0, 0, 0, 0, 0, 0, 0]",[rat]
2,My_pdf118new_method,[0 0 0 0 1 0 0 0 0],annot1,['dog'],\r\nMaterials and Methods\r\nAneurysm Construc...,"[0, 0, 0, 0, 0, 0, 1, 0, 0]",[dog]
3,My_pdf118new_title^abstract,[0 0 0 0 1 0 0 0 0],annot1,['dog'],Endovascular treatment of intracranial aneurys...,"[0, 0, 0, 0, 0, 0, 1, 0, 0]",[dog]
4,My_pdf127new_method,[0 1 0 0 0 0 0 0 0],annot1,['mouse'],\r\nMaterials and Methods\r\niPS cell cultures...,"[1, 0, 0, 0, 0, 0, 0, 0, 0]",[mouse]
...,...,...,...,...,...,...,...
1423,My_pdf860_title^abstract,[0 1 0 0 0 0 0 0 0],overlapping,['mouse'],Combined Nurr1 and Foxa2 roles in the therapy ...,"[1, 0, 0, 0, 0, 0, 0, 0, 0]",[mouse]
1424,My_pdf876_method,[1 0 0 0 0 0 0 0 0],overlapping,['rat'],\r\n2 Material and methods\r\n2.1 Animal and r...,"[0, 1, 0, 0, 0, 0, 0, 0, 0]",[rat]
1425,My_pdf876_title^abstract,[1 0 0 0 0 0 0 0 0],overlapping,['rat'],Efficacy of Danhong Injection on Serum Concent...,"[0, 1, 0, 0, 0, 0, 0, 0, 0]",[rat]
1426,My_pdf881_method,[0 1 0 0 0 0 0 0 0],overlapping,['mouse'],\r\nMaterials and Methods\r\nReagents\r\nThe a...,"[1, 0, 0, 0, 0, 0, 0, 0, 0]",[mouse]


In [172]:
evaluate_multilabel_predictions(species_annotated)


Micro  - Precision: 0.892, Recall: 0.971, F1: 0.930
Macro  - Precision: 0.671, Recall: 0.924, F1: 0.742

Classification Report:

               precision    recall  f1-score   support

          cat       0.13      1.00      0.23         6
          dog       0.68      1.00      0.81        15
   guinea pig       0.67      1.00      0.80         6
       monkey       0.82      0.67      0.74        21
        mouse       0.91      0.98      0.95       539
          pig       0.56      1.00      0.71        10
       rabbit       0.85      1.00      0.92        28
          rat       0.96      0.98      0.97       815
species-other       0.47      0.69      0.56        29

    micro avg       0.89      0.97      0.93      1469
    macro avg       0.67      0.92      0.74      1469
 weighted avg       0.92      0.97      0.94      1469
  samples avg       0.93      0.97      0.94      1469



{'labels': ['cat',
  'dog',
  'guinea pig',
  'monkey',
  'mouse',
  'pig',
  'rabbit',
  'rat',
  'species-other'],
 'precision_micro': 0.8918073796122576,
 'recall_micro': 0.9707283866575902,
 'f1_micro': 0.9295958279009127,
 'precision_macro': 0.6714181711838244,
 'recall_macro': 0.924101118631023,
 'f1_macro': 0.7419886814948478,
 'report': '               precision    recall  f1-score   support\n\n          cat       0.13      1.00      0.23         6\n          dog       0.68      1.00      0.81        15\n   guinea pig       0.67      1.00      0.80         6\n       monkey       0.82      0.67      0.74        21\n        mouse       0.91      0.98      0.95       539\n          pig       0.56      1.00      0.71        10\n       rabbit       0.85      1.00      0.92        28\n          rat       0.96      0.98      0.97       815\nspecies-other       0.47      0.69      0.56        29\n\n    micro avg       0.89      0.97      0.93      1469\n    macro avg       0.67      0.

In [174]:
def compute_f1(precision, recall):
    """Compute F1 score from precision and recall."""
    if (precision + recall) == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

In [180]:
compute_f1(0.92, 0.93)

0.924972972972973

In [6]:
270 + 120

390